# Legoin – Tahap 1: Deteksi Kepingan LEGO dengan YOLO11

**Kelompok 5 – Visual Komputer Cerdas** (Dika Aprilio Wibowo, Aldio Lisafron, Ricky Bonardo)

Notebook ini melatih **YOLO11** untuk mendeteksi setiap kepingan LEGO dalam satu foto. Model hanya punya **1 kelas (`lego`)**; jenis part dikenali di tahap 2. Hasil deteksi dipakai untuk:
1. **menghitung jumlah kepingan** (dasar inventori), dan
2. **memotong (crop) tiap kepingan** sebagai input tahap 2 (klasifikasi jenis part dengan MobileNetV3).

**Dataset:** Zawora, K., Zaraziński, S., Śledź, B., & Łobacz, B. (2021). *Tagged images with LEGO bricks*. MOST Wiedzy, Gdańsk University of Technology. https://doi.org/10.34808/anq4-rn44 (lisensi CC BY 4.0, wajib dicantumkan sumbernya).

- 2.933 foto nyata berisi 1–32 kepingan (latar dan pencahayaan acak) + 2.908 render (1 kepingan, latar putih)
- Anotasi bounding box format **PASCAL VOC** (XML, dibuat dengan labelImg)
- Struktur zip: `photos/<jumlah kepingan>/*.jpg|xml` dan `renders/1/*.jpg|xml`

**Runtime yang disarankan:** Google Colab dengan GPU (*Runtime → Change runtime type → T4 GPU*). Ukuran zip ±6 GB.

**Alur:** Setup → Unduh data → Eksplorasi → Konversi VOC → YOLO → Split → Training → Evaluasi → Uji hitung kepingan → Crop untuk tahap 2 → Simpan model

## 1. Setup lingkungan

In [ ]:
# Versi dikunci supaya hasil antar anggota kelompok sama
%pip install -q ultralytics==8.4.190 pandas tqdm

In [ ]:
import os
import sys
from pathlib import Path

# Google Colab: data mentah & olahan di disk Colab (cepat), hasil training di Google Drive (permanen).
# Lokal (VS Code/Jupyter): semua disimpan di folder notebook ini.
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content')
    SIMPAN_DIR = Path('/content/drive/MyDrive/VISKOM_Kelompok5')
else:
    ROOT = Path.cwd()
    SIMPAN_DIR = ROOT
SIMPAN_DIR.mkdir(parents=True, exist_ok=True)

RAW_DIR = ROOT / 'data' / 'lego_tagged_raw'   # hasil unzip dataset asli
YOLO_DIR = ROOT / 'data' / 'lego_yolo'        # dataset format YOLO hasil konversi
RUNS_DIR = SIMPAN_DIR / 'runs'                # log & bobot hasil training

In [ ]:
import torch
import ultralytics

print('ultralytics', ultralytics.__version__, '| torch', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'tidak ada (training di CPU akan sangat lambat)')

## 2. Konfigurasi

Semua angka yang biasa diubah saat eksperimen dikumpulkan di sini.

In [ ]:
SEED = 42
MODEL_AWAL = 'yolo11n.pt'          # n = paling ringan; coba 'yolo11s.pt' bila GPU cukup
EPOCHS = 50
IMGSZ = 640                        # coba 960 bila kepingan kecil sering terlewat
BATCH = 16                         # turunkan ke 8 bila GPU kehabisan memori
PATIENCE = 15                      # early stopping bila mAP val tidak membaik
UKURAN_MAKS = 1280                 # sisi terpanjang foto setelah di-resize (foto asli s.d. 6000x8000 px)
PAKAI_RENDER = True                # render hanya dimasukkan ke data train
RASIO_SPLIT = (0.70, 0.15, 0.15)   # train / val / test, khusus foto nyata
NAMA_KELAS = ['lego']

## 3. Unduh dan ekstrak dataset

Link MOST Wiedzy di bawah mengarah (redirect) ke file `02 - tagged1.zip` di S3. `wget -c` bisa melanjutkan unduhan yang terputus. Cell ini dilewati bila dataset sudah pernah diekstrak.

In [ ]:
URL_DATASET = ('https://mostwiedzy.pl/en/open-research-data/'
               'tagged-images-with-lego-bricks,202309140833448152311-0/download')
ZIP_PATH = ROOT / 'data' / 'lego_tagged.zip'

if (RAW_DIR / 'photos').exists():
    print('Dataset sudah ada di', RAW_DIR)
else:
    ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
    !wget -c -q --show-progress -O "{ZIP_PATH}" "{URL_DATASET}"
    !unzip -q -o "{ZIP_PATH}" -d "{RAW_DIR}"
    print('Selesai ekstrak ke', RAW_DIR)
    # ZIP_PATH.unlink()  # aktifkan bila disk hampir penuh

## 4. Eksplorasi data

In [ ]:
import xml.etree.ElementTree as ET
from collections import Counter

import pandas as pd


def baca_voc(xml_path):
    """Baca ukuran gambar dan daftar box (nama, xmin, ymin, xmax, ymax) dari XML PASCAL VOC."""
    root = ET.parse(xml_path).getroot()
    size = root.find('size')
    w = int(float(size.findtext('width', '0'))) if size is not None else 0
    h = int(float(size.findtext('height', '0'))) if size is not None else 0
    boxes = []
    for obj in root.iter('object'):
        bb = obj.find('bndbox')
        boxes.append((obj.findtext('name', '').strip(),
                      *(float(bb.findtext(k)) for k in ('xmin', 'ymin', 'xmax', 'ymax'))))
    return w, h, boxes


rows = []
for xml_path in sorted(RAW_DIR.rglob('*.xml')):
    rel = xml_path.relative_to(RAW_DIR)
    img_path = next((p for p in (xml_path.with_suffix(e) for e in ('.jpg', '.jpeg', '.png', '.JPG'))
                     if p.exists()), None)
    w, h, boxes = baca_voc(xml_path)
    rows.append({'xml': xml_path, 'img': img_path, 'sumber': rel.parts[0], 'folder_n': int(rel.parts[1]),
                 'n_box': len(boxes), 'xml_w': w, 'xml_h': h, 'nama': [b[0] for b in boxes]})
df = pd.DataFrame(rows)
df = df[df.img.notna()].reset_index(drop=True)   # buang XML yang gambarnya tidak ada

print(df.groupby('sumber').size().rename('jumlah gambar'))
print('\nNama label di XML:', Counter(n for ns in df.nama for n in ns).most_common(10))
df.head()

Nama label di XML tidak seragam (ada salah ketik), tetapi semuanya berarti "kepingan LEGO". Karena itu semua box dipetakan ke satu kelas `lego`.

In [ ]:
import matplotlib.pyplot as plt

foto = df[df.sumber == 'photos']
print('Jumlah box sama dengan nama folder:', f'{(foto.folder_n == foto.n_box).mean():.1%}')

ax = foto.n_box.value_counts().sort_index().plot.bar(figsize=(10, 3), color='tab:blue')
ax.set_xlabel('jumlah kepingan per foto')
ax.set_ylabel('jumlah foto')
ax.set_title('Sebaran jumlah kepingan per foto nyata')
plt.show()

Sebagian besar foto hanya berisi 1 kepingan, padahal aplikasi kita memotret banyak kepingan sekaligus. Karena itu split data nanti dibuat **bertingkat (stratified)** menurut jumlah kepingan, supaya data val dan test tetap berisi foto multi-kepingan.

### 4.1 Cek orientasi EXIF

Foto dari HP menyimpan arah putar di tag EXIF *Orientation*. Bila koordinat label dibuat pada piksel mentah tetapi gambar dibaca dalam keadaan sudah diputar (atau sebaliknya), box akan meleset. Cell berikut membandingkan ukuran di XML dengan ukuran gambar untuk menentukan frame koordinat setiap label.

In [ ]:
from PIL import Image, ImageOps


def cek_frame(r):
    with Image.open(r.img) as im:
        w, h = im.size
    if (r.xml_w, r.xml_h) == (w, h):
        return 'mentah'      # label mengikuti piksel mentah
    if (r.xml_w, r.xml_h) == (h, w):
        return 'diputar'     # label mengikuti gambar yang sudah diputar sesuai EXIF
    return 'beda'            # ukuran XML lain; koordinat diskalakan proporsional


df['frame'] = [cek_frame(r) for r in df.itertuples()]
df.groupby(['sumber', 'frame']).size().rename('jumlah')

In [ ]:
from PIL import ImageDraw


def muat_gambar(r, maks=None):
    """Buka gambar dalam frame yang sama dengan koordinat XML, opsional di-resize ke sisi terpanjang `maks`.

    Mengembalikan (gambar RGB, skala_x, skala_y) untuk mengubah koordinat XML ke piksel gambar ini.
    """
    im = Image.open(r.img)
    w0, h0 = im.size
    if r.frame == 'diputar':
        w0, h0 = h0, w0
    k = min(1.0, maks / max(w0, h0)) if maks else 1.0
    if k < 1:
        target = (round(w0 * k), round(h0 * k))
        # decode JPEG langsung di resolusi lebih kecil, jauh lebih cepat untuk foto 12+ MP
        im.draft('RGB', target[::-1] if r.frame == 'diputar' else target)
    if r.frame == 'diputar':
        im = ImageOps.exif_transpose(im)
    im = im.convert('RGB')
    if k < 1:
        im = im.resize(target, Image.LANCZOS)
    ref_w, ref_h = (r.xml_w, r.xml_h) if r.xml_w > 0 and r.xml_h > 0 else (w0, h0)
    return im, im.width / ref_w, im.height / ref_h


kandidat = df[(df.sumber == 'photos') & (df.n_box >= 3)]
contoh = kandidat.sample(min(6, len(kandidat)), random_state=SEED)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax in axes.flat:
    ax.axis('off')
for ax, r in zip(axes.flat, contoh.itertuples()):
    im, sx, sy = muat_gambar(r, maks=800)
    draw = ImageDraw.Draw(im)
    for _, x1, y1, x2, y2 in baca_voc(r.xml)[2]:
        draw.rectangle([x1 * sx, y1 * sy, x2 * sx, y2 * sy], outline=(255, 0, 0), width=4)
    ax.imshow(im)
    ax.set_title(f'{r.n_box} kepingan')
plt.tight_layout()
plt.show()

## 5. Split train / val / test

Foto nyata dibagi 70/15/15 secara bertingkat menurut jumlah kepingan (1, 2, 3–5, 6+). Render (latar putih, 1 kepingan) hanya masuk ke **train**, supaya evaluasi murni dilakukan pada foto nyata.

In [ ]:
import random

rng = random.Random(SEED)


def strata(n):
    return '1' if n <= 1 else '2' if n == 2 else '3-5' if n <= 5 else '6+'


split = {}
for _, g in df[df.sumber == 'photos'].groupby(df.n_box.map(strata)):
    idx = list(g.index)
    rng.shuffle(idx)
    n_tr, n_va = round(len(idx) * RASIO_SPLIT[0]), round(len(idx) * RASIO_SPLIT[1])
    split.update({i: 'train' for i in idx[:n_tr]})
    split.update({i: 'val' for i in idx[n_tr:n_tr + n_va]})
    split.update({i: 'test' for i in idx[n_tr + n_va:]})
if PAKAI_RENDER:
    split.update({i: 'train' for i in df[df.sumber == 'renders'].index})
df['split'] = df.index.map(split)
df['strata'] = df.n_box.map(strata)
pd.crosstab([df.sumber, df.strata], df["split"], margins=True)

## 6. Konversi PASCAL VOC → format YOLO

Format label YOLO: satu baris per objek, `kelas x_tengah y_tengah lebar tinggi`, semuanya dinormalisasi 0–1 terhadap ukuran gambar.

Sekaligus:
- foto di-resize ke sisi terpanjang `UKURAN_MAKS` (training jauh lebih cepat, disk lebih hemat),
- gambar disimpan ulang **tanpa EXIF** supaya orientasinya pasti sama dengan label,
- box yang terpotong tepi gambar di-clip, box yang terlalu kecil dibuang.

In [ ]:
import shutil
from concurrent.futures import ThreadPoolExecutor

from tqdm.auto import tqdm


def ke_yolo(r):
    """Simpan gambar + label YOLO untuk satu baris df, kembalikan jumlah box yang ditulis."""
    nama = f'{r.sumber}_{r.folder_n}_{r.img.stem}'
    im, sx, sy = muat_gambar(r, maks=UKURAN_MAKS)
    W, H = im.size
    baris = []
    for _, x1, y1, x2, y2 in baca_voc(r.xml)[2]:
        x1, x2 = sorted((x1 * sx, x2 * sx))
        y1, y2 = sorted((y1 * sy, y2 * sy))
        x1, y1, x2, y2 = max(0, x1), max(0, y1), min(W, x2), min(H, y2)
        if x2 - x1 < 2 or y2 - y1 < 2:
            continue
        baris.append(f'0 {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} '
                     f'{(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}')
    im.save(YOLO_DIR / 'images' / r.split / f'{nama}.jpg', quality=92)
    (YOLO_DIR / 'labels' / r.split / f'{nama}.txt').write_text(''.join(b + '\n' for b in baris))
    return len(baris)


if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)
for s in ('train', 'val', 'test'):
    (YOLO_DIR / 'images' / s).mkdir(parents=True)
    (YOLO_DIR / 'labels' / s).mkdir(parents=True)

dipakai = df[df["split"].notna()]
with ThreadPoolExecutor(max_workers=os.cpu_count()) as ex:
    n_tulis = list(tqdm(ex.map(ke_yolo, dipakai.itertuples()), total=len(dipakai), desc='konversi'))
df.loc[dipakai.index, 'n_box_yolo'] = n_tulis
df.groupby('split').agg(gambar=('img', 'size'), box=('n_box_yolo', 'sum'))

In [ ]:
import yaml

data_yaml = YOLO_DIR / 'data.yaml'
data_yaml.write_text(yaml.safe_dump({
    'path': str(YOLO_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': dict(enumerate(NAMA_KELAS)),
}, sort_keys=False))
print(data_yaml.read_text())

## 7. Training YOLO11

Bobot awal `yolo11n.pt` (pretrained COCO) diunduh otomatis, lalu di-fine-tune ke kelas `lego` (transfer learning). Augmentasi bawaan Ultralytics (mosaic, flip, HSV, scale) sudah aktif.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_AWAL)
model.train(data=str(data_yaml), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE,
            seed=SEED, project=str(RUNS_DIR), name='yolo11_lego', exist_ok=True)

TRAIN_DIR = Path(model.trainer.save_dir)
BEST = TRAIN_DIR / 'weights' / 'best.pt'
print('Model terbaik:', BEST)

In [ ]:
from IPython.display import Image as IPImage
from IPython.display import display

display(IPImage(filename=str(TRAIN_DIR / 'results.png'), width=1000))

## 8. Evaluasi pada data test

Angka di bawah dicatat ke laporan sebagai baseline deteksi.

In [ ]:
model_best = YOLO(BEST)
m = model_best.val(data=str(data_yaml), split='test', imgsz=IMGSZ, batch=BATCH,
                   project=str(RUNS_DIR), name='uji_test', exist_ok=True)

metrik_deteksi = pd.Series({
    'mAP@50': m.box.map50,
    'mAP@50-95': m.box.map,
    'Precision': m.box.mp,
    'Recall': m.box.mr,
}).round(3)
metrik_deteksi

In [ ]:
for f in sorted(Path(m.save_dir).glob('*PR_curve.png'))[:1]:
    display(IPImage(filename=str(f), width=600))

## 9. Uji hitung kepingan (metrik inventori)

Untuk aplikasi, yang paling penting adalah **jumlah kepingan per foto benar**. Threshold confidence dipilih pada data **val** (yang memaksimalkan akurasi hitung tepat), lalu dilaporkan pada data **test**:
- **MAE** = rata-rata selisih absolut jumlah prediksi vs jumlah sebenarnya
- **Akurasi hitung tepat** = persentase foto yang jumlah prediksinya persis benar

In [ ]:
import numpy as np

THRESHOLDS = np.round(np.arange(0.10, 0.91, 0.05), 2)


def kumpulkan_conf(split):
    """Jalankan prediksi sekali (conf rendah), simpan confidence tiap box + jumlah label sebenarnya."""
    files = sorted((YOLO_DIR / 'images' / split).glob('*.jpg'))
    gt = np.array([sum(1 for l in (YOLO_DIR / 'labels' / split / f'{f.stem}.txt').read_text().splitlines()
                       if l.strip()) for f in files])
    confs = [res.boxes.conf.cpu().numpy()
             for res in model_best.predict([str(f) for f in files], imgsz=IMGSZ, conf=0.05,
                                           stream=True, verbose=False)]
    return files, gt, confs


def skor_hitung(gt, confs, t):
    pred = np.array([(c >= t).sum() for c in confs])
    return np.abs(pred - gt).mean(), (pred == gt).mean(), pred


_, gt_val, conf_val = kumpulkan_conf('val')
kurva = pd.DataFrame([(t, *skor_hitung(gt_val, conf_val, t)[:2]) for t in THRESHOLDS],
                     columns=['threshold', 'MAE', 'akurasi_tepat'])
CONF_TERBAIK = float(kurva.loc[kurva.akurasi_tepat.idxmax(), 'threshold'])

ax = kurva.plot(x='threshold', y=['MAE', 'akurasi_tepat'], marker='o', figsize=(8, 3))
ax.axvline(CONF_TERBAIK, color='gray', ls='--')
ax.set_title(f'Pemilihan threshold pada data val (terbaik = {CONF_TERBAIK})')
plt.show()

In [ ]:
files_test, gt_test, conf_test = kumpulkan_conf('test')
mae, akurasi, pred_test = skor_hitung(gt_test, conf_test, CONF_TERBAIK)
print(f'Threshold {CONF_TERBAIK} -> MAE {mae:.2f} kepingan | akurasi hitung tepat {akurasi:.1%}')

hasil_hitung = pd.DataFrame({'strata': [strata(n) for n in gt_test], 'gt': gt_test, 'pred': pred_test})
hasil_hitung['selisih'] = (hasil_hitung["pred"] - hasil_hitung["gt"]).abs()
hasil_hitung.groupby('strata').agg(foto=('gt', 'size'), MAE=('selisih', 'mean'),
                                   akurasi_tepat=('selisih', lambda s: (s == 0).mean())).round(3)

## 10. Contoh prediksi + crop untuk tahap 2

`save_crop=True` menyimpan potongan setiap kepingan yang terdeteksi. Potongan ini bisa dipakai untuk menguji classifier tahap 2 (jenis part) pada data hasil deteksi, bukan hanya gambar bersih.

In [ ]:
multi = [f for f in files_test if int(f.name.split('_')[1]) >= 3] or files_test
pilih = multi[:6]
hasil_pred = model_best.predict([str(f) for f in pilih], imgsz=IMGSZ, conf=CONF_TERBAIK, save_crop=True,
                                project=str(RUNS_DIR), name='prediksi_contoh', exist_ok=True, verbose=False)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax in axes.flat:
    ax.axis('off')
for ax, res in zip(axes.flat, hasil_pred):
    ax.imshow(res.plot()[:, :, ::-1])   # plot() mengembalikan BGR
    ax.set_title(f'{len(res.boxes)} kepingan terdeteksi')
plt.tight_layout()
plt.show()

crop_dir = Path(model_best.predictor.save_dir) / 'crops'
print(len(list(crop_dir.rglob('*.jpg'))), 'crop tersimpan di', crop_dir)

## 11. Simpan dan ekspor model

In [ ]:
tujuan = SIMPAN_DIR / 'legoin_yolo11_best.pt'
shutil.copy(BEST, tujuan)
print('Bobot terbaik disalin ke', tujuan)

# Opsional: ekspor ke ONNX untuk aplikasi web (onnxruntime-web) atau backend tanpa PyTorch
EKSPOR_ONNX = False
if EKSPOR_ONNX:
    print(model_best.export(format='onnx', imgsz=IMGSZ))

## 12. Catatan dan langkah berikutnya

- Catat ke laporan: mAP@50, mAP@50-95, Precision, Recall (bagian 8) serta MAE dan akurasi hitung tepat per strata (bagian 9).
- Eksperimen yang layak dibandingkan:
  - `PAKAI_RENDER = True` vs `False`, untuk melihat pengaruh data render (sejalan dengan temuan Vidal dkk., 2023 dan Boiński, 2023)
  - `yolo11n` vs `yolo11s`, serta `IMGSZ` 640 vs 960 untuk foto yang berisi banyak kepingan kecil
- Tambahkan **foto mandiri kelompok** (LEGO di meja/lantai rumah) ke folder `images/` dan `labels/` dengan format yang sama, lalu fine-tune dari `best.pt`.
- Tahap berikutnya: crop kepingan → **MobileNetV3** (jenis part) → **K-Means HSV** (warna) → inventori JSON → **AI agent**.